# LATTE on Attar: timing-driven detailed placement, step by step

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/latte_on_attar.ipynb)

**LATTE** (*Legality-Assured differentiable Timing-driven detailed placement*, DAC 2026) improves the timing of a
legal placement by moving its most critical cells a little. Each round it:

1. times the placement exactly and builds a **differentiable timer** that matches it there,
2. takes the cells whose position matters most for TNS (the largest gradient),
3. splits them into **independent sets** (no two cells of a set share a net),
4. moves each set by gradient descent on TNS, WNS and wirelength, then legalizes it,
5. keeps the round only if the exact timing improved.

[Attar](https://github.com/RayChang987/Attar) supplies everything around the algorithm: the design, a static timer
equal to OpenSTA, a global router whose routes give the parasitics the paper measures with, incremental
legalization and legality checks. What you read below is only LATTE itself, about 600 lines, split into the
pieces of the paper.

**Runtime:** a CPU runtime is enough (about 5 minutes for 12 rounds on Colab's two cores); a GPU runtime also
works.

## 0. Setup

In [ ]:
!pip install -q attar==1.0.5

In [ ]:
import base64
import gzip
import math
import time

import numpy as np
import torch

import attar
from attar import Evaluator, naive
from attar.timing.sta import nldm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('attar', attar.__version__, '| torch', torch.__version__, '| device', DEVICE)

## 1. The design, the ruler and the starting placement

The design is `des` from the ICCAD 2025 contest (2.5k cells, ASAP7), downloaded from Attar's zoo on first use.
Before clock-tree synthesis the clock is timed as **ideal**, with the per-register clock latencies the contest SDC
gives (useful skew), as OpenSTA does pre-CTS. `d.ideal_clocks()` switches the design to that.

In [ ]:
DESIGN = 'des'
d = attar.load('after_fp/iccad2025/' + DESIGN, device=DEVICE)
d.ideal_clocks()
print('%s: %d cells (%d movable), %d nets' % (DESIGN, len(d.names['cells']), int(d.movable.sum()), len(d.n_ptr) - 1))

**The ruler.** The paper judges a placement by its timing after an early global route. In Attar that is two
calls: route the nets, then time the design with the parasitics of those routes. Every result below, and LATTE's
own accept / revert decision, uses this function.

In [ ]:
def evaluate(d):
    routes, st = d.global_route(verbose=False)
    g = d.timing(routes=routes).summary()
    out = {'legal': bool(Evaluator(d).legality()['legal']), 'hpwl': float(d.hpwl()),
           'wl': float(st['wirelength_um']), 'wns': g['wns'], 'tns': g['tns']}
    d.timing()
    return out

**The start.** Like the paper, LATTE starts from a DREAMPlace placement (global placement, legalization and
detailed placement). DREAMPlace is not installed here, so its placement of `des` is embedded in the next cell
(Bookshelf `.pl`, compressed).

In [ ]:
# DREAMPlace's placement of des (Bookshelf .pl, gzip + base64)
START_PL = '...'

In [ ]:
def set_start(d):
    idx = {n.replace('\\', ''): i for i, n in enumerate(d.names['cells'])}
    ii, xy = [], []
    for line in gzip.decompress(base64.b64decode(START_PL)).decode().splitlines():
        t = line.split()
        if len(t) >= 3 and not line.startswith(('UCLA', '#')) and t[0] in idx and bool(d.movable[idx[t[0]]]):
            ii.append(idx[t[0]]); xy.append((float(t[1]) / d.dbu, float(t[2]) / d.dbu))
    p = d.pos.clone()
    p[torch.as_tensor(ii, device=d.device)] = torch.as_tensor(xy, device=d.device, dtype=p.dtype)
    d.pos = p
    if not Evaluator(d).legality()['legal']:          # rows' rail orientations: repaired in place
        naive.legalize(d, verbose=False, incremental=True)
    assert Evaluator(d).legality()['legal']
    return len(ii)


print('DREAMPlace start: %d cells placed' % set_start(d))
pos0, ori0 = d.pos.clone(), d.orient.clone()
e0 = evaluate(d)
print('start  GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s' % (e0['wl'], e0['wns'], e0['tns'], e0['legal']))

## 2. A differentiable timer anchored on the exact one (paper §3.2)

LATTE needs the gradient of TNS with respect to every cell's position. Instead of a second, approximate STA, the
timer here is **anchored** on Attar's exact timing at the current placement `pos0`. Every arc delay and slew is

$$d(\text{pos}) = d_\text{exact}(\text{pos}_0) + d_\text{model}(\text{pos}) - d_\text{model}(\text{pos}_0),$$

so at `pos0` the timer equals OpenSTA exactly, and nearby it moves with a simple model whose gradient is cheap:
Elmore delay on an RC tree per net and the cells' NLDM tables. Required times and the clock network are held at
their `pos0` values (detailed placement does not move the clock).

### 2.1 Wires: an RC tree per net

Each data net becomes a tree rooted at its driver: a rectilinear spanning tree (Prim's algorithm on Manhattan
distance) built once per anchor. Its topology then stays fixed while the cells move, so the Elmore delay to every
sink, the net's capacitance and its length are smooth functions of the pin positions. Nets are grouped into
buckets by degree so that all trees of a bucket are evaluated together as batched matrix products: `anc` marks
the nodes on each root-to-node path, `ch` each node's children.

In [ ]:
INF = float('inf')
K_SLEW = math.log(9.0)          # 10-90 % slew of a ramp through an RC stage (PERI)
BUCKETS = (2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024)


def _smooth_abs(x, eps=1e-3):
    return torch.sqrt(x * x + eps * eps)


class Trees:
    """Rooted RC trees of the data nets (fixed topology between rebuilds): per degree bucket the pins (driver first),
    each node's parent, the ancestor matrix (u on the path root -> v, u != root) and the child matrix."""

    def __init__(self, d, nets, drivers, pin_cap, pins_xy):
        dev = d.device
        self.dev = dev
        P = len(pin_cap)
        p_net = d.p_net.cpu().numpy()
        order = np.argsort(p_net, kind='stable')
        ptr = np.searchsorted(p_net[order], np.arange(len(d.n_ptr)))
        deg = ptr[1:] - ptr[:-1]
        self.buckets = []
        xy = pins_xy.detach()
        nets = np.asarray(nets)
        dn = deg[nets]
        sizes = sorted(set(b for b in BUCKETS if b < int(dn.max(initial=2))) | {max(int(dn.max(initial=2)), 2)})
        lo = 0
        for n in sizes:
            sel = nets[(dn > lo) & (dn <= n)]
            if len(sel):
                self.buckets.append(self._bucket(sel, n, ptr, order, drivers, xy, pin_cap, P))
            lo = n

    def _bucket(self, sel, n, ptr, order, drivers, xy, pin_cap, P):
        dev = self.dev
        B = len(sel)
        pins = np.full((B, n), P, np.int64)                      # P: a dummy pin (zero position, zero cap)
        cnt = ptr[sel + 1] - ptr[sel]
        row = np.repeat(np.arange(B), cnt)
        col = np.arange(int(cnt.sum())) - np.repeat(np.cumsum(cnt) - cnt, cnt)
        flat = order[np.repeat(ptr[sel], cnt) + col]
        pins[row, col] = flat
        # the driver to slot 0 (swap)
        dv = drivers[sel]
        where = np.nonzero(pins == dv[:, None])
        slot = np.zeros(B, np.int64); slot[where[0]] = where[1]
        first = pins[:, 0].copy()
        pins[np.arange(B), slot] = first
        pins[:, 0] = dv
        pins = torch.as_tensor(pins, device=dev)
        valid = pins < P
        X = torch.cat([xy, torch.zeros(1, 2, device=dev, dtype=xy.dtype)])[pins]          # (B, n, 2)
        with torch.no_grad():                                    # Prim's tree, all nets of the bucket at once
            D = (X[:, :, None, :] - X[:, None, :, :]).abs().sum(-1)
            D = D.masked_fill(~valid[:, None, :] | ~valid[:, :, None], INF)
            intree = torch.zeros(B, n, dtype=torch.bool, device=dev); intree[:, 0] = True
            best = D[:, 0].clone(); par = torch.zeros(B, n, dtype=torch.long, device=dev)
            ins = torch.zeros(B, n, dtype=torch.long, device=dev)    # insertion order
            ar = torch.arange(B, device=dev)
            for k in range(1, n):
                cand = best.masked_fill(intree, INF)
                j = cand.argmin(1)
                ok = torch.isfinite(cand[ar, j])
                j = torch.where(ok, j, torch.zeros_like(j))
                ins[:, k] = j
                intree[ar[ok], j[ok]] = True
                dj = D[ar, j]
                better = (dj < best) & ~intree & ok[:, None]
                best = torch.where(better, dj, best)
                par = torch.where(better, j[:, None].expand(-1, n), par)
            anc = torch.zeros(B, n, n, device=dev, dtype=xy.dtype)
            for k in range(1, n):
                v = ins[:, k]
                live = v > 0
                row = anc[ar, par[ar, v]].clone()
                row[ar, v] = 1.0
                anc[ar[live], v[live]] = row[live]
            nonroot = valid.clone(); nonroot[:, 0] = False
            ch = torch.zeros(B, n, n, device=dev, dtype=xy.dtype)
            bi, wi = torch.nonzero(nonroot, as_tuple=True)
            ch[bi, par[bi, wi], wi] = 1.0
        cap = torch.cat([pin_cap, torch.zeros(1, device=dev, dtype=pin_cap.dtype)])[pins].clone()
        cap[:, 0] = 0.0                                          # the driver's own pin is not a load
        return {'nets': torch.as_tensor(sel, device=dev), 'pins': pins, 'par': par, 'anc': anc, 'ch': ch,
                'nonroot': nonroot.to(xy.dtype), 'cap': cap * valid}

With the topology fixed, evaluating a bucket is a few batched products. For every edge into a node, its length L
(a smoothed Manhattan distance, so the gradient exists at zero); each node carries its pin capacitance plus half of
the wire capacitance of the edges it touches; the capacitance downstream of a node is a product with the ancestor
matrix, and the Elmore delay of a node is the sum, over the edges on its path from the root, of the edge's
resistance times the capacitance downstream of it.

In [ ]:
def eval_trees(self, pins_xy, r, c, n_nets, P):
    """Elmore delay per pin (P,), total capacitance per net (E,), wirelength per net (E,), differentiable."""
    xy = torch.cat([pins_xy, torch.zeros(1, 2, device=pins_xy.device, dtype=pins_xy.dtype)])
    m1_all, c_all, l_all, pin_all, net_all = [], [], [], [], []
    for b in self.buckets:
        X = xy[b['pins']]
        Xp = torch.gather(X, 1, b['par'][:, :, None].expand(-1, -1, 2))
        L = _smooth_abs(X - Xp).sum(-1) * b['nonroot']                       # (B, n) edge into each node
        nodecap = b['cap'] + 0.5 * c * (L + torch.bmm(b['ch'], L[:, :, None])[:, :, 0])
        cdown = torch.bmm(b['anc'].transpose(1, 2), nodecap[:, :, None])[:, :, 0]
        m1 = torch.bmm(b['anc'], (r * L * cdown)[:, :, None])[:, :, 0]
        m1_all.append(m1.flatten()); pin_all.append(b['pins'].flatten())
        c_all.append(nodecap.sum(1)); l_all.append(L.sum(1)); net_all.append(b['nets'])
    m1p = torch.zeros(P + 1, device=xy.device, dtype=xy.dtype).index_add(0, torch.cat(pin_all), torch.cat(m1_all))[:P]
    nets = torch.cat(net_all)
    cn = torch.zeros(n_nets, device=xy.device, dtype=xy.dtype).index_add(0, nets, torch.cat(c_all))
    ln = torch.zeros(n_nets, device=xy.device, dtype=xy.dtype).index_add(0, nets, torch.cat(l_all))
    return m1p, cn, ln


Trees.eval = eval_trees

### 2.2 Anchoring: what the timer takes from Attar's exact timing

`anchor()` runs the ruler (global route + timing) and keeps, from Attar's timer, everything the model holds fixed
or starts from:

* the arrival `A0` and slew `S0` of every pin and the load `C0` of every driver;
* the **levels** of the timing graph in propagation order: the clock's launch arcs first, then each level's cell
  arcs (from pin, to pin, Liberty arc, transitions) and the wire arcs leaving it, with their exact delays;
* the setup required time of every endpoint, plus a per-endpoint offset (`ep_off`) that absorbs whatever the model
  leaves out (CRPR credit, latch borrowing), so that the model's slacks equal the exact ones at `pos0`.

This part reads Attar timer fields that are not a public API (`T.cb_lev`, `T._rec`, ...); Attar 1.0.5 also
exposes the same data read-only through `TimingResult.graph()`, `library()` and `arrivals()`.

In [ ]:
class DSTA:
    """Differentiable setup timing of design d, anchored at the current placement (exact timer run there)."""

    def __init__(self, d, max_degree=1024, dtype=torch.float32):
        self.d = d
        self.max_degree = max_degree
        self.dt = dtype
        self.anchor()

    def exact(self, pos=None):
        """Exact timing on the ruler (Attar global routes + their parasitics) at pos (default: the placement)."""
        d = self.d
        routes, st = d.global_route(pos=pos, verbose=False)
        res = d.timing(pos=pos, routes=routes)
        return res, d._gr_timer[1], st

    def anchor(self, pos=None):
        d = self.d
        if pos is not None:
            d.pos = pos
        res, T, self.gr_stats = self.exact()
        self.T, self.res = T, res
        dev = d.device
        dt = self.dt
        self.dev = dev
        P, NT = T.P, T.NT
        self.P, self.NT = P, NT
        self.pos0 = d.pos.detach().clone()
        self.A0 = T.amax.reshape(P * 2, NT).detach().clone().to(dt)
        self.S0 = T.smax.reshape(-1).detach().clone().to(dt)
        self.C0 = T.load.detach().clone().to(dt)                    # (P, 2) total load per driver pin, rise / fall
        self.pin_cap = T.pin_cap.to(dt).mean(1)
        self.r, self.c = float(T.r_sig), float(T.c_sig)
        # the RC trees of the data nets (the clock network stays as timed)
        clk = T.clk_of >= 0
        drv = T.net_driver
        p_net = d.p_net.cpu().numpy()
        deg = np.bincount(p_net, minlength=len(drv))
        nets = np.nonzero((drv >= 0) & (deg >= 2) & (deg <= self.max_degree))[0]
        nets = nets[~clk[drv[nets]]]
        self.model_nets = nets
        self.p_net_t = d.p_net
        self.trees = Trees(d, nets, drv, self.pin_cap, self._pins(self.pos0.to(dt)))
        with torch.no_grad():
            self.m1_0, self.cn_0, _ = self.trees.eval(self._pins(self.pos0.to(dt)), self.r, self.c, len(drv), P)
        # the levels: the ideal clock's launch arcs first (T.lc), then the levels of the search
        recs = list(T._rec)
        stages = []
        if T.lc is not None:
            lib, to, orf, tag, t0, cks = T.lc
            ea, en = T.lc_ex
            stages.append((dict(lib=lib, f=None, t_=to, orf=orf, irf=None, ea=ea, en=en,
                                arr=torch.ones(len(lib), dtype=torch.bool, device=dev), tag=tag, t0=t0, cks=cks),
                           recs.pop(0)))
        for cb in T.cb_lev:
            if cb is not None:
                lib, f, t_, orf, irf, ea, en, arr = cb
                stages.append((dict(lib=lib, f=f, t_=t_, orf=orf, irf=irf, ea=ea, en=en, arr=arr), recs.pop(0)))
        assert not recs
        self.levels = [self._level(c, rec) for c, rec in stages]
        # nets driven by input ports: their sinks move with the wire only
        pa = T.pa
        self.pa_rows = torch.cat([T.n_to[pa] * 2, T.n_to[pa] * 2 + 1])
        self.pa_sink = torch.cat([T.n_to[pa], T.n_to[pa]])
        # endpoints: required times at pos0; CRPR and other credits as a constant offset per pin
        rows, reqs = [], []
        for rws, req, _ in T._setup_required():
            rows.append(rws); reqs.append(torch.where(torch.isnan(req), torch.full_like(req, INF), req).to(self.dt))
        self.ep_rows = torch.cat(rows); self.ep_req = torch.cat(reqs)
        self.ep_pin = self.ep_rows // 2
        self.ep_u, self.ep_inv = torch.unique(self.ep_pin, return_inverse=True)
        mine0 = self._slack_pins(self.A0)
        exact = res.setup.to(self.dt)[self.ep_u]
        fin = torch.isfinite(exact) & torch.isfinite(mine0)
        self.ep_ok = fin
        self.ep_idx = torch.nonzero(fin).flatten()
        self.ep_off = torch.where(fin, exact - mine0, torch.zeros_like(exact))
        self.exact_tns, self.exact_wns = res.tns, res.wns
        return self

The rest of the class: turning one level of Attar's search into the tensors `slacks()` needs (its exact delays
`dg0`, `wx0` and the model's own values at `pos0`, `dm0`, `sm0`, which the delta form subtracts), and small
helpers for pin positions, NLDM table lookups and endpoint slacks.

In [ ]:
class DSTA(DSTA):
    def _level(self, c, rec):
        """One level of the search: its cell arcs, the wire arcs leaving it and their exact delays at pos0."""
        dt, dev = self.dt, self.dev
        T = self.T
        lib, t_, orf, ea, en, arr = c['lib'], c['t_'], c['orf'], c['ea'], c['en'], c['arr']
        launch = c['f'] is None
        fi = None if launch else c['f'] * 2 + c['irf']
        ti = t_ * 2 + orf
        tu, g_inv = torch.unique(ti, return_inverse=True)
        sk = T.n_to[en]
        si = sk * 2 + orf[ea]
        su, l_inv = torch.unique(si, return_inverse=True)
        C0 = self.C0[t_, orf]
        s_in0 = c['cks'].to(dt) if launch else self.S0[fi]
        dm0 = self._lookup(lib, orf, s_in0, C0)
        sm0 = self._lookup(lib, 2 + orf, s_in0, C0)
        smod0 = torch.full((len(tu),), -INF, device=dev, dtype=dt).scatter_reduce(0, g_inv, sm0, 'amax')
        L = dict(tab_d=self._tables(lib, orf), tab_s=self._tables(lib, 2 + orf), lib=lib, fi=fi, ti=ti, t=t_, orf=orf,
                 tu=tu, g_inv=g_inv, ea=ea, sk=sk, su=su, l_inv=l_inv, arr=arr, dg0=rec[0].detach().clone().to(dt),
                 wx0=rec[1].detach().clone().to(dt), dm0=dm0, sm0=sm0, smod0=smod0, C0=C0,
                 net_t=self.p_net_t[t_], launch=launch)
        if launch:
            L.update(tag=c['tag'], t0=c['t0'].to(dt), s_in=s_in0)
        return L

    # -- small helpers ---------------------------------------------------------------------------------------------
    def _pins(self, pos):
        """Pin locations at pos, with the orientations of the last sync_orient() (a buffer the timer owns: a CUDA
        graph keeps reading it; d.update_orient() replaces d.pin_off with a new tensor)."""
        if not hasattr(self, '_pin_off') or self._pin_off.dtype != pos.dtype:
            self._pin_off = self.d.pin_off.to(pos.dtype).clone()
            self._half = (self.d.size[self.d.p_cell] / 2).to(pos.dtype)
        elif not (pos.is_cuda and torch.cuda.is_current_stream_capturing()):
            self.sync_orient()
        return pos[self.d.p_cell] + self._half + self._pin_off

    def sync_orient(self):
        if hasattr(self, '_pin_off'):
            self._pin_off.copy_(self.d.pin_off)

    def _tables(self, lib, slot):
        T = self.T
        return (T.t_ix1[lib, slot].to(self.dt).contiguous(), T.t_ix2[lib, slot].to(self.dt).contiguous(),
                T.t_val[lib, slot].to(self.dt).contiguous(), T.t_n[lib, slot].contiguous())

    def _lookup(self, lib, slot, x, y):
        return nldm(*self._tables(lib, slot), x, y)

    def _slack_pins(self, A):
        s = (self.ep_req - A[self.ep_rows]).amin(1)
        s = torch.where(torch.isfinite(s), s, torch.full_like(s, INF))
        return torch.full((len(self.ep_u),), INF, device=self.dev, dtype=s.dtype).scatter_reduce(
            0, self.ep_inv, s, 'amin')

### 2.3 Forward: slacks, level by level

`slacks(pos)` propagates setup arrivals through the levels, as a graph-based (GBA) max over the incoming arcs,
for each transition and clock tag:

* a **cell arc's** delay is its exact delay plus the change of its NLDM table value, as the driver's load changes
  with the wires (`dcn`: the change of the net's tree capacitance) and the input slew changes;
* a **wire arc's** delay is its exact delay plus the change of the sink's Elmore delay (`dm1`), and the sink's slew
  is degraded by the wire as $\sqrt{s^2 + (\ln 9 \cdot m_1)^2}$;
* an endpoint's slack is its required time minus its arrival, plus `ep_off`.

TNS is the sum of the negative slacks, WNS the worst one; both are differentiable (subgradients through the maxima).

In [ ]:
def slacks(self, pos):
    """Endpoint setup slacks (n,) at pos (differentiable), for the endpoints in self.ep_u[self.ep_ok]."""
    pos = pos.to(self.dt)
    pins = self._pins(pos)
    m1, cn, _ = self.trees.eval(pins, self.r, self.c, len(self.T.net_driver), self.P)
    dm1 = m1 - self.m1_0
    dcn = cn - self.cn_0
    A, S = self.A0, self.S0
    # nets driven by input ports
    rows = self.pa_rows
    A = A.index_copy(0, rows, self.A0[rows] + dm1[self.pa_sink][:, None])
    for L in self.levels:
        # cell arcs into this level's pins
        fi = L['fi']
        s_in = L['s_in'] if L['launch'] else S[fi]
        C = L['C0'] + dcn[L['net_t']]
        dg = L['dg0'] + nldm(*L['tab_d'], s_in, C) - L['dm0']
        sm = nldm(*L['tab_s'], s_in, C)
        arr = L['arr'][:, None]
        NT = A.shape[1]
        if L['launch']:
            base = torch.full((len(dg), NT), -INF, device=self.dev, dtype=A.dtype)
            cand = base.scatter(1, L['tag'][:, None], (L['t0'] + dg)[:, None])
        else:
            cand = torch.where(arr, A[fi] + dg[:, None], torch.full_like(A[fi], -INF))
        An = torch.full((len(L['tu']), NT), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
            0, L['g_inv'][:, None].expand(-1, NT), cand, 'amax')
        A = A.index_copy(0, L['tu'], An)
        smod = torch.full((len(L['tu']),), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
            0, L['g_inv'], sm, 'amax')
        S = S.index_copy(0, L['tu'], self.S0[L['tu']] + smod - L['smod0'])
        # wire arcs leaving them
        if len(L['ea']):
            ea, sk = L['ea'], L['sk']
            w = L['wx0'] + dm1[sk]
            cl = torch.where(arr[ea], cand[ea] + w[:, None], torch.full_like(cand[ea], -INF))
            Ln = torch.full((len(L['su']), NT), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                0, L['l_inv'][:, None].expand(-1, NT), cl, 'amax')
            A = A.index_copy(0, L['su'], Ln)
            sd = S[L['ti'][ea]]; sd0 = self.S0[L['ti'][ea]]
            dsl = torch.sqrt(sd * sd + (K_SLEW * m1[sk]) ** 2) - torch.sqrt(sd0 * sd0 + (K_SLEW * self.m1_0[sk]) ** 2)
            dsu = torch.full((len(L['su']),), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                0, L['l_inv'], dsl, 'amax')
            S = S.index_copy(0, L['su'], self.S0[L['su']] + dsu)
    s = self._slack_pins(A) + self.ep_off
    return s.index_select(0, self.ep_idx)


def forward(self, pos):
    s = self.slacks(pos)
    return s.clamp(max=0).sum(), s.min().clamp(max=0)          # TNS, WNS


def cell_grad(self, pos=None):
    """G_i = dTNS / d(x_i, y_i) at pos (default: the anchor), (C, 2)."""
    pos = (self.pos0 if pos is None else pos).detach().to(self.dt).requires_grad_()
    tns, _ = self.forward(pos)
    g, = torch.autograd.grad(tns, pos)
    return g, float(tns)


DSTA.slacks, DSTA.forward, DSTA.cell_grad = slacks, forward, cell_grad

### 2.4 Check: exact at the anchor, a gradient everywhere

At `pos0` the differentiable timer must give exactly the ruler's TNS and WNS. Its gradient then says which cells
matter: here are the five cells with the largest $\|\partial \text{TNS} / \partial (x, y)\|$.

In [ ]:
ds = DSTA(d)
with torch.no_grad():
    tns_m, wns_m = ds.forward(ds.pos0)
print('exact (ruler):        TNS %10.2f ps   WNS %7.2f ps' % (ds.exact_tns, ds.exact_wns))
print('model at the anchor:  TNS %10.2f ps   WNS %7.2f ps' % (float(tns_m), float(wns_m)))
g, _ = ds.cell_grad()
top = torch.topk(g.norm(dim=1), 5).indices
for c in top.tolist():
    print('  %-28s dTNS/dx %8.2f  dTNS/dy %8.2f  ps/um' % (d.names['cells'][c], float(g[c, 0]), float(g[c, 1])))

### 2.5 Speed: the objective as one CUDA graph

Within a round the trees and levels do not change shape, so on a GPU the whole objective, -TNS - λ1·WNS and its
gradient over all cells, is captured once as a CUDA graph and replayed at every step. On a CPU it simply runs
eagerly.

In [ ]:
class GraphedObjective:
    """-TNS - lam1 WNS and its gradient over all cell positions. call(pos) -> (tns, wns, grad (C, 2))."""

    def __init__(self, ds, lam1):
        self.ds, self.lam1 = ds, lam1
        self.static = ds.pos0.to(ds.dt).clone().requires_grad_()
        self.graph = None
        if ds.dev.type != 'cuda':
            return
        st = torch.cuda.Stream(device=ds.dev)
        st.wait_stream(torch.cuda.current_stream(ds.dev))
        with torch.cuda.stream(st):                                # warm up, then capture
            for _ in range(2):
                tns, wns = ds.forward(self.static)
                g, = torch.autograd.grad(-tns - lam1 * wns, self.static)
        torch.cuda.current_stream(ds.dev).wait_stream(st)
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.tns, self.wns = ds.forward(self.static)
            self.g, = torch.autograd.grad(-self.tns - lam1 * self.wns, self.static)

    def __call__(self, pos):
        if self.graph is None:
            p = pos.detach().to(self.ds.dt).requires_grad_()
            tns, wns = self.ds.forward(p)
            g, = torch.autograd.grad(-tns - self.lam1 * wns, p)
            return tns.detach(), wns.detach(), g
        with torch.no_grad():
            self.static.copy_(pos)
            self.ds.sync_orient()
        self.graph.replay()
        return self.tns, self.wns, self.g

## 3. LATTE (paper §3.3-3.6)

### 3.1 Independent sets (Algorithm 1)

Cells that share a net pull on each other's timing, so moving them together would make the gradient stale.
LATTE colors the critical cells greedily, most critical first, so that no two cells of a set share a net (clock
nets and nets of fanout above 64 do not count), then balances the set sizes. The sets are optimized one after
another, most critical first.

In [ ]:
def adjacency_nets(d, fanout_max, T):
    """The nets that make two cells adjacent: no clock nets, fanout <= fanout_max."""
    p_net = d.p_net
    deg = torch.bincount(p_net, minlength=len(d.n_ptr) - 1)
    drv = torch.as_tensor(T.net_driver, device=d.device)
    clk = torch.as_tensor(T.clk_of >= 0, device=d.device)
    clk_net = torch.zeros_like(deg, dtype=torch.bool)
    ok = drv >= 0
    clk_net[ok] = clk[drv[ok]]
    return (deg >= 2) & (deg <= fanout_max + 1) & ~clk_net


def independent_sets(cells, score, d, keep_net, T_bal):
    """Algorithm 1: greedy coloring by score (descending), then balancing. cells, score: numpy."""
    p_cell = d.p_cell.cpu().numpy(); p_net = d.p_net.cpu().numpy(); keep = keep_net.cpu().numpy()
    nets_of = {c: set() for c in cells.tolist()}
    m = np.isin(p_cell, cells) & keep[p_net]
    for c, n in zip(p_cell[m].tolist(), p_net[m].tolist()):
        nets_of[c].add(n)
    order = cells[np.argsort(-score, kind='stable')]
    sc = dict(zip(cells.tolist(), score.tolist()))
    sets, used = [], []
    for c in order.tolist():                                       # greedy coloring
        for k in range(len(sets)):
            if not (nets_of[c] & used[k]):
                sets[k].append(c); used[k] |= nets_of[c]
                break
        else:
            sets.append([c]); used.append(set(nets_of[c]))
    for _ in range(T_bal):                                         # balancing: largest set -> smallest
        sizes = [len(s) for s in sets]
        i, j = int(np.argmax(sizes)), int(np.argmin(sizes))
        if sizes[i] - sizes[j] <= 1:
            break
        moved = False
        for c in sorted(sets[i], key=lambda c: sc[c]):
            if not (nets_of[c] & used[j]):
                sets[i].remove(c); used[i] = set().union(*[nets_of[x] for x in sets[i]]) if sets[i] else set()
                sets[j].append(c); used[j] |= nets_of[c]
                moved = True
                break
        if not moved:
            break
    sets = [s for s in sets if s]
    sets.sort(key=lambda s: -max(sc[c] for c in s))
    return sets


# the sets of this anchor's 1 % most critical cells
score = g.norm(dim=1); score[~d.movable] = 0
crit = torch.topk(score, max(20, int(0.01 * int(d.movable.sum())))).indices
sets = independent_sets(crit.cpu().numpy(), score[crit].cpu().numpy(), d, adjacency_nets(d, 64, ds.T), 1000)
print('%d critical cells -> %d independent sets of sizes %s' % (len(crit), len(sets), [len(s) for s in sets]))

### 3.2 Moving one set: Adam with CAWP (paper §3.5)

The cells of a set move by Adam on

$$F = -\text{TNS} - \lambda_1 \text{WNS} + \lambda_2 \text{WL},$$

with every other cell fixed. λ2 is set at the first step so that the wirelength gradient is a fixed fraction of
the timing gradient. Two details from the paper:

* **CAWP** (criticality-aware weight preconditioning): each cell's gradient is divided by the root mean square of
  its first gradients, so cells with weak timing gradients still move;
* a **warmup + cosine** learning rate, and each cell stays within `radius` µm of where the round started.

In [ ]:
def warmup_cosine(t, n, warm, lr):
    if t < warm:
        return lr * (t + 1) / warm
    return lr * 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, n - warm)))


def optimize_set(ds, d, S, cfg, base_pos, log, obj):
    """Adam + CAWP on the cells S (index tensor); other cells fixed at base_pos. Returns the new positions of S."""
    xy0 = base_pos[S].to(ds.dt)
    xy = xy0.clone().requires_grad_()
    opt = torch.optim.Adam([xy], lr=cfg['lr'])
    hist = []
    lam2 = None
    pfull = base_pos.to(ds.dt).clone()
    for t in range(cfg['iters']):
        for g_ in opt.param_groups:
            g_['lr'] = warmup_cosine(t, cfg['iters'], cfg['warmup'], cfg['lr'])
        with torch.no_grad():
            pfull[S] = xy
        tns, wns, gall = obj(pfull)                               # timing: -TNS - lam1 WNS
        g_t = gall[S].clone()
        pw = pfull.detach().float().requires_grad_()               # wirelength (Attar's WA model)
        wl = d.wa_wirelength(pw, gamma=0.5)
        g_w = torch.autograd.grad(wl, pw)[0][S].to(ds.dt)
        if lam2 is None:
            nt, nw = float(g_t.norm()), float(g_w.norm())
            lam2 = cfg['lam2'] * (nt / nw if nw > 0 and nt > 0 else 0.0)
        g = g_t + lam2 * g_w
        if cfg['cawp']:                                            # CAWP (Eq. 4)
            if len(hist) < cfg['T0']:
                hist.append((g * g).sum(1))
            h = torch.stack(hist).mean(0)
            pre = cfg['tau'] / torch.sqrt(h + cfg['eps'])
            g = g * pre[:, None]
        xy.grad = g
        opt.step()
        with torch.no_grad():
            dlt = (xy - xy0).clamp(-cfg['radius'], cfg['radius'])
            xy.copy_(xy0 + dlt)
        if t == 0 or t == cfg['iters'] - 1:
            log.append({'t': t, 'tns': float(tns), 'wns': float(wns), 'wl': float(wl)})
    return xy.detach()

### 3.3 A round, and the outer loop

A round anchors a new timer at the current legal placement, takes the `k_frac` most critical cells by gradient,
splits them into sets and, set by set, moves them and **legalizes them incrementally**:
`naive.legalize(d, cells=mask)` moves only the cells in `mask` to legal spots, as the paper's §3.6 does.
Then the ruler decides: the round is kept if TNS improved (or TNS tied and WNS improved), otherwise the placement
goes back and the round's cells sit out until a round is accepted.

The log line of a round reports the timer's predicted TNS gain before and after legalization, which shows how much
of the gradient step survives legalization. First the parameters, with the paper's defaults:

In [ ]:
DEFAULTS = dict(
    rounds=30,              # maximum rounds
    patience=3,             # stop after this many rounds without improvement
    k_frac=0.02,            # critical cells per round, fraction of the movable cells
    k_min=20,
    fanout_max=64,          # nets above this fanout do not make two cells adjacent (Algorithm 1)
    balance_T=1000,         # balancing iterations (Algorithm 1)
    iters=40,               # Adam iterations per set
    lr=0.15,                # peak learning rate (um per step)
    warmup=5,               # warmup iterations, then cosine annealing to 0
    tau=1.0, eps=1e-12, T0=10,   # CAWP (Eq. 4)
    lam1=1.0,               # WNS weight (Eq. 3)
    lam2=0.3,               # WL weight, relative to |grad TNS| / |grad WL| at a set's first iteration
    radius=6.0,             # max move of a cell from its round-start position (um)
    max_sets=64,            # sets beyond this wait for a later round
    cawp=True,
    time_limit=7200.0,
)

The loop itself: anchor, select, split, move and legalize set by set, then let the ruler decide.

In [ ]:
def ruler_summary(d):
    routes, _ = d.global_route(verbose=False)
    return d.timing(routes=routes).summary()


def run(d, cfg=None, verbose=True):
    cfg = dict(DEFAULTS, **(cfg or {}))
    t_start = time.time()
    ev = Evaluator(d)
    mv = torch.nonzero(d.movable).flatten()
    K = max(cfg['k_min'], int(cfg['k_frac'] * len(mv)))
    xmin = float(d.pos[mv, 0].min()); xmax = float((d.pos[mv, 0] + d.size[mv, 0]).max())    # the core box
    ymin = float(d.pos[mv, 1].min()); ymax = float((d.pos[mv, 1] + d.size[mv, 1]).max())
    best = ruler_summary(d)
    hist = [{'round': 0, 'tns': best['tns'], 'wns': best['wns'], 'hpwl': float(d.hpwl()), 'accepted': True, 't': 0.0}]
    if verbose:
        print('start  TNS %.1f WNS %.2f HPWL %.1f' % (best['tns'], best['wns'], hist[0]['hpwl']), flush=True)
    keep_net = None
    bad = 0
    tabu = set()                       # cells of rejected rounds sit out until a round is accepted
    for r in range(1, cfg['rounds'] + 1):
        if time.time() - t_start > cfg['time_limit']:
            break
        tr = time.time()
        # 1. anchor the differentiable timer, select the critical cells, split them into sets
        ds = DSTA(d)
        if keep_net is None:
            keep_net = adjacency_nets(d, cfg['fanout_max'], ds.T)
        pos_r = d.pos.clone(); ori_r = d.orient.clone()
        g, tns_d = ds.cell_grad()
        score = g.norm(dim=1)
        score[~d.movable] = 0
        if tabu:
            score[torch.as_tensor(sorted(tabu), device=d.device)] = 0
        k = min(K, int((score > 0).sum()))
        if k == 0:
            break
        crit = torch.topk(score, k).indices
        sets = independent_sets(crit.cpu().numpy(), score[crit].cpu().numpy(), d, keep_net, cfg['balance_T'])
        sets = sets[:cfg['max_sets']]
        # 2. move and legalize the sets one by one
        base = d.pos.clone()
        logs = []
        diag = {'pre': 0.0, 'post': 0.0, 'move_um': 0.0, 'lg_um': 0.0}
        set_t = [torch.as_tensor(S_, device=d.device) for S_ in sets]
        obj = GraphedObjective(ds, cfg['lam1'])
        for S in set_t:
            xy = optimize_set(ds, d, S, cfg, base, logs, obj)
            xy[:, 0] = torch.minimum(xy[:, 0].clamp(min=xmin), xmax - d.size[S, 0].to(xy.dtype))
            xy[:, 1] = torch.minimum(xy[:, 1].clamp(min=ymin), ymax - d.size[S, 1].to(xy.dtype))
            with torch.no_grad():                                  # predicted gain before legalization
                t_b = float(ds.forward(base)[0])
                pre = base.clone(); pre[S] = xy.to(pre.dtype)
                t_pre = float(ds.forward(pre)[0])
            p = d.pos.clone(); p[S] = xy.float(); d.pos = p
            mask = torch.zeros(d.pos.shape[0], dtype=torch.bool, device=d.device); mask[S] = True
            naive.legalize(d, cells=mask, verbose=False)            # incremental: only the set moves
            with torch.no_grad():                                  # ... and after it
                t_post = float(ds.forward(d.pos)[0])
            diag['pre'] += t_pre - t_b; diag['post'] += t_post - t_b
            diag['move_um'] += float((xy.float() - base[S]).abs().sum(1).mean()) / len(sets)
            diag['lg_um'] += float((d.pos[S] - xy.float()).abs().sum(1).mean()) / len(sets)
            base = d.pos.clone()
        # 3. the ruler decides
        res = ruler_summary(d)
        acc = res['tns'] > best['tns'] + 1e-6 or (res['tns'] >= best['tns'] - 1e-6 and res['wns'] > best['wns'])
        legal = bool(ev.legality()['legal'])
        acc = acc and legal
        if acc:
            best = res; bad = 0
            tabu.clear()
        else:
            d.pos = pos_r; d.orient = ori_r; d.update_orient()
            bad += 1
            tabu.update(crit.tolist())
        h = {'round': r, 'tns': res['tns'], 'wns': res['wns'], 'hpwl': float(d.hpwl()), 'accepted': acc,
             'legal': legal, 'k': k, 'sets': len(sets), 'dsta_tns': tns_d, 't': time.time() - t_start,
             'round_s': time.time() - tr, 'diag': diag}
        hist.append(h)
        if verbose:
            print('round %2d  TNS %.1f WNS %.2f HPWL %.1f  %-6s k=%d sets=%d  %4.1fs | predicted gain %.1f -> %.1f '
                  'after LG, moves %.2f um, LG %.2f um' % (r, res['tns'], res['wns'], h['hpwl'],
                                                          'acc' if acc else 'REVERT', k, len(sets), h['round_s'],
                                                          diag['pre'], diag['post'], diag['move_um'], diag['lg_um']),
                  flush=True)
        if bad >= cfg['patience']:
            break
    d.timing()
    return hist

## 4. Run LATTE

The configuration is the one of the full reproduction; only the number of rounds is cut for Colab (the full runs
allow up to 200 rounds in 2 hours; 12 rounds give most of the gain on `des`).

In [ ]:
ROUNDS = 12
cfg = {'rounds': ROUNDS, 'patience': 40, 'lr': 0.5, 'k_frac': 0.01, 'lam1': 50, 'lam2': 0.1, 'time_limit': 20 * 60}
t = time.time()
hist = run(d, cfg)
rt = time.time() - t
e1 = evaluate(d)
pos_latte = d.pos.clone()
print('\nLATTE  GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s   (%.0f s)'
      % (e1['wl'], e1['wns'], e1['tns'], e1['legal'], rt))

## 5. Against a wirelength-driven detailed placer

The baseline is Attar's detailed placer (`naive.detailed_place`: OpenROAD dpo's moves, wirelength only), from the
same DREAMPlace start.

In [ ]:
d.pos, d.orient = pos0.clone(), ori0.clone(); d.update_orient()
t = time.time()
naive.detailed_place(d, verbose=False)
e2 = evaluate(d)
print('A-DP   GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s   (%.0f s)'
      % (e2['wl'], e2['wns'], e2['tns'], e2['legal'], time.time() - t))
d.pos = pos_latte

In [ ]:
print('%-28s %10s %9s %11s %9s %9s %9s %6s' % ('', 'GR WL (um)', 'WNS (ps)', 'TNS (ps)', 'WL ratio', 'WNS ratio',
                                              'TNS ratio', 'legal'))
for name, e in (('start (DREAMPlace)', e0), ('A-DP (Attar, WL-driven)', e2), ('LATTE, %d rounds' % (len(hist) - 1), e1)):
    print('%-28s %10.0f %9.2f %11.1f %9.3f %9.3f %9.3f %6s' % (name, e['wl'], e['wns'], e['tns'], e['wl'] / e0['wl'],
                                                               e['wns'] / e0['wns'], e['tns'] / e0['tns'], e['legal']))
print('\nfor reference (ratios to the start):')
for name, r in (('our full run on des (200 rounds)', (1.007, 0.938, 0.886)),
                ('our full run, 8-design geo-mean', (1.006, 0.928, 0.861)), ('paper, geo-mean', (1.041, 0.884, 0.868))):
    print('%-34s WL %.3f   WNS %.3f   TNS %.3f' % (name, *r))

## 6. Convergence and what moved

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
r = [h['round'] for h in hist]; tns = [h['tns'] for h in hist]
a0 = ax[0]
a0.plot(r, np.maximum.accumulate(tns), color='#2a6fdb', lw=2)
acc = [h for h in hist[1:] if h['accepted']]; rej = [h for h in hist[1:] if not h['accepted']]
a0.scatter([h['round'] for h in acc], [h['tns'] for h in acc], s=36, color='#2a6fdb', zorder=3, label='accepted round')
a0.scatter([h['round'] for h in rej], [h['tns'] for h in rej], s=36, facecolor='white', edgecolor='#888',
           zorder=3, label='reverted round (TNS got worse)')
a0.axhline(e2['tns'], color='#c0504d', ls='--', lw=1)
a0.text(r[0], e2['tns'], 'A-DP (WL-driven DP)', color='#c0504d', va='bottom')
a0.set_xlabel('round'); a0.set_ylabel('TNS after global routing (ps)'); a0.set_title('LATTE on %s' % DESIGN)
a0.grid(alpha=.3); a0.legend(frameon=False, loc='lower right')

a1 = ax[1]
p0, p1 = pos0.cpu().numpy(), pos_latte.cpu().numpy()
mv = np.abs(p1 - p0).sum(1) > 1e-6
dx = p1[mv] - p0[mv]
a1.scatter(p0[:, 0], p0[:, 1], s=1, color='#d0d0d0')
a1.scatter(p0[mv, 0], p0[mv, 1], s=14, facecolor='white', edgecolor='#2a6fdb', zorder=3, label='before')
a1.scatter(p1[mv, 0], p1[mv, 1], s=14, color='#2a6fdb', zorder=4, label='after')
a1.quiver(p0[mv, 0], p0[mv, 1], dx[:, 0], dx[:, 1], angles='xy', scale_units='xy', scale=1, width=0.004,
          color='#2a6fdb', zorder=4)
a1.legend(frameon=False, loc='upper right')
a1.set_aspect('equal'); a1.set_title('%d cells moved, mean %.2f um (true scale)' % (mv.sum(), np.abs(dx).sum(1).mean()))
a1.set_xlabel('x (um)'); a1.set_ylabel('y (um)')
plt.tight_layout(); plt.show()

## Notes

* **Full reproduction** (8 ICCAD 2025 IWLS designs, DREAMPlace start, up to 2 h each): TNS geo-mean 0.861 against
  the paper's 0.868 (OpenROAD's own global route + OpenSTA give 0.892), wirelength +0.6 % against the paper's
  +4.1 %. LATTE beats wirelength-driven detailed placers, DREAMPlace 4.0's timing-driven placement and Rsyn there;
  those baselines need native tools and are not run here.
* Cells move only a fraction of a micron: on these ASAP7 designs timing is mostly cell delay, so the paper's
  larger wirelength increase does not appear.
* Other designs: set `DESIGN` to one of `ac97_top aes des3 fpu mc_top pci_bridge32 tv80s` and start from Attar's
  own placement (`naive.global_place(d)`, then `naive.legalize(d)`) instead of `set_start`; use a GPU runtime for
  the bigger ones.